# Brain Tumor CNN: Simple Kaggle Notebook

## Step 0: Add the dataset

Add this Kaggle dataset as notebook input:

[Brain MRI Images for Brain Tumor Detection](https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection)

Choose **Notebook options → Accelerator → GPU** in Kaggle before training.

> This is a learning project only. Do not use it for medical diagnosis.

## Step 1: Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

## Step 2: Check that a GPU is available (optional)

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", len(gpus) > 0)
print(gpus)

## Step 3: Load the images

In [ ]:
# Direct path after adding the linked Kaggle dataset as notebook input.
DATA_DIR = Path(
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/"
    "brain_tumor_dataset"
)
if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Add the Brain MRI Images for Brain Tumor Detection dataset from the link in Step 0."
    )
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

class_names = train_ds.class_names
print("Classes:", class_names)  # ['no', 'yes']

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

## Step 4: Create a simple CNN

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),

    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## Step 5: Train and save the model

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)
model.save("/kaggle/working/brain_tumor_cnn.keras")

## Step 6: Check the accuracy

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print("Validation accuracy:", round(accuracy * 100, 2), "%")

plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Step 7: Test one image

In [ ]:
# Select any image. Change 'yes' to 'no' to test a no-tumor image.
TEST_IMAGE_PATH = next((DATA_DIR / "yes").glob("*.jpg"))

image = tf.keras.utils.load_img(TEST_IMAGE_PATH, target_size=IMAGE_SIZE)
image_array = tf.keras.utils.img_to_array(image)
image_array = np.expand_dims(image_array, axis=0)

prediction = model.predict(image_array, verbose=0)[0]
predicted_class = class_names[np.argmax(prediction)]
confidence = np.max(prediction) * 100

plt.imshow(image)
plt.axis("off")
plt.title(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.show()

print("Prediction:", predicted_class)
print("Confidence:", round(confidence, 2), "%")